In [0]:
# =====================================
# IMPORT REQUIRED LIBRARIES
# =====================================
import dlt
from pyspark.sql.functions import *

# =====================================
# STEP 1: JOIN ALL TABLES
# =====================================
@dlt.table(
    name="business_sales_enriched",
    comment="Joined data from sales, customers, and products"
)
def business_sales_enriched():

    sales = spark.readStream.table("sales_stream").alias("s")
    customers = spark.read.table("customers").alias("c")
    products = spark.read.table("products").alias("p")

    df = (
        sales
        .join(customers, col("s.customer_id") == col("c.customer_id"), "inner")
        .join(products, col("s.product_id") == col("p.product_id"), "inner")
        .select(
            col("s.sales_id"),
            col("s.customer_id"),
            col("s.product_id"),
            col("s.total"),
            col("s.sale_timestamp"),
            col("c.customer_name"),
            col("c.region"),
            col("p.category")
        )
    )

    return df


# =====================================
# STEP 2: AGGREGATION
# =====================================
@dlt.table(
    name="sales_by_region",
    comment="Aggregated sales by region and category"
)
def sales_by_region():

    dfagg = spark.readStream.table("business_sales_enriched")

    return (
        dfagg
        .groupBy("region", "category")
        .agg(sum("total").alias("total_sales"))
    )